In [5]:
!pip install -U openai langchain-community langchain-huggingface langchain-text-splitters faiss-cpu beautifulsoup4 pypdf
import os
from openai import OpenAI
from google.colab import userdata

# LangChain Imports
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Step 1: Ingesting data from multiple sources...")

# SOURCE A: Hardcoded Text / Paragraphs (The Detective Data)
raw_paragraphs = [
    """• Establishment & Management: Founded in 1999, CIT is sponsored by the Kodava Education Society and managed under a dedicated governing council.
• Affiliations: The institution is approved by the AICTE, affiliated with Visvesvaraya Technological University (VTU) Belagavi, and recognized by the Government of Karnataka.
• Campus & Location: The college features a 31-acre pollution-free campus situated in Halligattu near Ponnampet."""
]
# Convert raw Python strings into LangChain Document objects
text_documents = [Document(page_content=text) for text in raw_paragraphs]

# SOURCE B: Web Scraping (The College Data)
urls = ["https://www.citcoorg.edu.in/cv-admission-form/"]
web_loader = WebBaseLoader(urls)
web_documents = web_loader.load()

# SOURCE C: PDF Documents
pdf_path = "/content/Coorg_Institute_of_Technology_Information (1).pdf"
try:
    pdf_loader = PyPDFLoader(pdf_path)
    pdf_documents = pdf_loader.load()
    print("PDF loaded successfully.")
except Exception as e:
    print(f"Warning: PDF '{pdf_path}' not found in Colab. Skipping PDF ingestion.")
    pdf_documents = []

# MERGE ALL SOURCES
all_documents = text_documents + web_documents + pdf_documents
print(f"Total raw pages/documents loaded: {len(all_documents)}")

# Step 2: Data Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(all_documents)
print(f"Data chopped into {len(chunks)} searchable chunks.")

# Step 3: Embeddings & FAISS Vector Database
print("Building FAISS Vector Database...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

# Step 4: Initialize LLM using the restored GROQ_API_KEY secret name
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("dee11")
)

print("\n817 Unified Universal Bot Online. Ask about the crime scene, college, or PDF.")
print("-" * 60)

# Step 5: The Chat Loop
while True:
    user_query = input("\nYou: ")
    if user_query.lower() == 'exit':
        break

    # Semantic Search across ALL sources simultaneously
    relevant_chunks = vector_db.similarity_search(user_query, k=3)
    retrieved_context = "\n\n".join([doc.page_content for doc in relevant_chunks])

    system_prompt = f"""
    You are an AI assistant powered by a multi-source knowledge base.
    Answer the user's question using ONLY the provided context.
    If the context does not contain the answer, say "I do not have enough information."
    Do NOT guess.

    CONTEXT:
    {retrieved_context}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.0
    )

    print(f"AI: {response.choices[0].message.content}")

Step 1: Ingesting data from multiple sources...
PDF loaded successfully.
Total raw pages/documents loaded: 4
Data chopped into 22 searchable chunks.
Building FAISS Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


817 Unified Universal Bot Online. Ask about the crime scene, college, or PDF.
------------------------------------------------------------

You: cit
AI: Coorg Institute of Technology (CIT) is a private engineering institution located in Ponnampet, South Kodagu, Karnataka. It was founded in 1999 by the Kodava Education Society (KES) and is managed under a dedicated governing council. The campus spans 31 acres of pollution‑free, lush green land. CIT is approved by the AICTE, affiliated with Visvesvaraya Technological University (VTU) Belagavi, and recognized by the Government of Karnataka. The current principal is Dr. M. Basavaraj.

You: admission process
AI: **Admission Process (as described in the provided context)**  

1. **Application Submission**  
   - Eligible candidates are invited to submit their applications as soon as possible.  
   - The application window is closing soon, so early submission is recommended.

2. **Merit‑Based Branch Preferences**  
   - Branch preferences a